In [14]:
import os

In [15]:
%pwd

'c:\\Users\\Harsha vardhan\\OneDrive\\Desktop\\TextSummarization-Project'

In [16]:
os.chdir("../")

In [17]:
%pwd

'c:\\Users\\Harsha vardhan\\OneDrive\\Desktop'

In [18]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

In [19]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories

In [20]:
class ConfigurationManager:
    def __init__(self, config_file_path = CONFIG_FILE_PATH,
         params_filepath=PARAMS_FILE_PATH):
        

        self.config= read_yaml(config_file_path)
        self.params= read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config=self.config.data_transformation
        create_directories([config.root_dir])

        data_transformation_config=DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name=config.tokenizer_name
        )
        return data_transformation_config

In [21]:
import os
from textsummarizer.constants import *
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

In [22]:
class DataTransformation:
    def __init__(self, config:DataTransformationConfig):
        self.config=config
        self.tokenizer=AutoTokenizer.from_pretrained(config.tokenizer_name)

    def convert_examples_to_features(self, example):
        input_encodings=self.tokenizer(example['dialogue'], max_length=1024, padding="max_length", truncation=True)
        target_encodings=self.tokenizer(example['summary'], max_length=128, padding="max_length", truncation=True)

        return{
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }

    def convert(self):
        dataset_samsum=load_from_disk(self.config.data_path)
        dataset_samsum_pt=dataset_samsum.map(self.convert_examples_to_features, batched=True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir,"samsum_dataset"))


In [23]:
try:
    config=ConfigurationManager()
    data_transformation_config=config.get_data_transformation_config()
    data_transformation=DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e 

[2026-03-04 19:37:53,853: INFO: common: yaml file: C:\Users\Harsha vardhan\OneDrive\Desktop\TextSummarization-Project\config\config.yaml loaded successfully]
[2026-03-04 19:37:53,856: INFO: common: yaml file: C:\Users\Harsha vardhan\OneDrive\Desktop\TextSummarization-Project\params.yaml loaded successfully]
[2026-03-04 19:37:53,859: INFO: common: created directory at: artifacts]
[2026-03-04 19:37:53,861: INFO: common: created directory at: artifacts/data_transformation]
[2026-03-04 19:37:54,251: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-04 19:37:54,286: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-03-04 19:37:54,556: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.js

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 34308.72 examples/s]
